In [1]:
DATA_DIR = "../data"
PDF_PATH = f"{DATA_DIR}/1706.03762v7-2.pdf"
PERSIST_DIR = "../chroma_store"


In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF beolvasás
reader = PdfReader(PDF_PATH)
pages = [page.extract_text() or "" for page in reader.pages]
full_text = "\n".join(pages)

print("The whole text length", len(full_text))

# Feldarabolás chunkokra
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

chunks = splitter.split_text(full_text)
print("Number of chunks:", len(chunks))
print("First chunks first 300 character\n", chunks[0][:300])


The whole text length 39601
Number of chunks: 39
First chunk first 300 character
 Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

# Embedding modell betöltése (CPU-barát, könnyen cserélhető)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Új Chroma kliens létrehozása (persistent mód, hogy elmentse az indexet)
client = chromadb.PersistentClient(path=PERSIST_DIR)

# Új collection létrehozása (ha már létezik, töröljük előtte)
collection_name = "rag_docs"
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(name=collection_name)

# Embeddingek létrehozása a chunkokból
embeddings = embed_model.encode(chunks, convert_to_numpy=True)

# Dokumentumok feltöltése a collection-be
ids = [f"doc_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_id": i, "source": PDF_PATH} for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    metadatas=metadatas,
    embeddings=embeddings,
)

print("ChromaDB collection created, documents uploaded:", len(chunks))

ChromaDB collection created, documents uploaded: 39


In [4]:
def retrieve(query: str, k: int = 4):
    # Lekérdezés embedding
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    
    # ChromaDB keresés
    res = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Átalakítás olvasható formára
    docs = []
    for i in range(len(res["documents"][0])):
        docs.append({
            "text": res["documents"][0][i],
            "metadata": res["metadatas"][0][i],
            "score": res["distances"][0][i]
        })
    return docs

# Teszteljük egy kérdéssel
results = retrieve("How many layers are there in a encoder?", k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result: {i+1} ---")
    print("Score:", doc["score"])
    print("Chunk ID:", doc["metadata"]["chunk_id"])
    print("Text preview:", doc["text"][:300])


--- Result: 1 ---
Score: 0.7016187906265259
Chunk ID: 7
Text preview: respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual

--- Result: 2 ---
Score: 1.0205157995224
Chunk ID: 14
Text preview: for different layer types. n is the sequence length, d is the representation dimension, k is the kernel
size of convolutions and r the size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2 · d) O(1) O(1)
R

--- Result: 3 ---
Score: 1.1477288007736206
Chunk ID: 38
Text preview: sentence. We give two such examples above, from two different heads from the encoder self-attention
at layer 5 of 6. The heads clearly learned to perform different tasks.
15


In [5]:
class DummyLLM:
    def __init__(self, name: str = "dummy-llm"):
        self.name = name

    def generate(self, prompt: str) -> str:
        # Csak visszaadja a prompt egy részét, demonstráció céljából
        return f"[{self.name}] Short answer based on context:\n{prompt[:500]}"

llm = DummyLLM()



In [6]:
def answer_question(query: str, k: int = 3):
    # Retriever hívás
    hits = retrieve(query, k=k)
    context_texts = [h["text"] for h in hits]
    joined_context = "\n\n---\n\n".join(context_texts)

    # Prompt összeállítása
    prompt = (
        "Answer briefly and accurately to the user's question. "
        "use the given context.\n\n"
        f"Question:\n{query}\n\n"
        f"Context:\n{joined_context}\n\n"
        "Answer:"
    )

    return llm.generate(prompt)

# Teszteljük
print(answer_question("How many layers are there in a encoder?", k=3))

[dummy-llm] Short answer based on context:
Answer briefly and accurately to the user's question. use the given context.

Question:
How many layers are there in a encoder?

Context:
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed b


In [9]:
from langgraph.graph import StateGraph, END
from typing import Dict, Any


State = Dict[str, Any]

# Node-ok
def planner_node(state: State) -> State:
    query = state["query"]
    state["needs_retrieval"] = len(query) > 10
    return state

def searcher_node(state: State) -> State:
    if state.get("needs_retrieval"):
        hits = retrieve(state["query"], k=3)
        state["contexts"] = hits
    else:
        state["contexts"] = []
    return state

def responder_node(state: State) -> State:
    contexts = state.get("contexts", [])
    context_texts = [c["text"] for c in contexts]
    joined_context = "\n\n---\n\n".join(context_texts)

    prompt = (
        f"Question:\n{state['query']}\n\n"
        f"Context:\n{joined_context}\n\n"
        "Válasz:"
    )
    state["answer"] = llm.generate(prompt)
    return state


# Graph összeállítása
graph = StateGraph(State)

graph.add_node("planner", planner_node)
graph.add_node("searcher", searcher_node)
graph.add_node("responder", responder_node)

graph.add_edge("planner", "searcher")
graph.add_edge("searcher", "responder")
graph.add_edge("responder", END)

graph.set_entry_point("planner")

# Compile
app = graph.compile()

In [10]:
queries = [
    "How many layers are there in a encoder?",
]

for q in queries:
    final_state = app.invoke({"query": q})
    print("\n==========================")
    print("Question:", q)
    print("Answer:\n", final_state["answer"])


Question: How many layers are there in a encoder?
Answer:
 [dummy-llm] Short answer based on context:
Question:
How many layers are there in a encoder?

Context:
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm
